# 2.2 Properties of Matrix Operations

**Course:** Elementary Linear Algebra (Larson, 8e) 
**Chapter 2 — Matrices**

---

## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '../..'))

In [ ]:
from linalg_core.matrix import Matrix
from linalg_core.ops import add, scalar_mul, multiply, transpose
from linalg_core import EPSILON
import random

random.seed(42)


def random_matrix(rows, cols):
    """Generate a random matrix with entries in [-10, 10]."""
    data = [random.uniform(-10, 10) for _ in range(rows * cols)]
    return Matrix(rows, cols, data)


def matrices_equal(A, B):
    """Check if two matrices are equal within EPSILON tolerance."""
    if A.rows != B.rows or A.cols != B.cols:
        return False
    return all(abs(a - b) < EPSILON for a, b in zip(A.data, B.data))


def is_zero_matrix(M):
    """Check if a matrix is the zero matrix within EPSILON tolerance."""
    return all(abs(x) < EPSILON for x in M.data)


def det2x2(M):
    """Compute the determinant of a 2x2 matrix (ad - bc).
    
    This is a preview helper; the full determinant lives in Chapter 3.
    """
    if M.rows != 2 or M.cols != 2:
        raise ValueError("Only 2x2 matrices supported")
    return M[0, 0] * M[1, 1] - M[0, 1] * M[1, 0]


print("Setup complete.")

---

## 2. Motivation

Matrix algebra looks like regular algebra — addition is commutative, multiplication is associative.
But some familiar rules **break**. Can we still factor? Can we cancel?

In ordinary arithmetic, if $ab = 0$ then $a = 0$ or $b = 0$. And if $ab = ac$ with $a \neq 0$,
we can cancel $a$ to get $b = c$. These are so natural we rarely think about them.

With matrices, **neither of these is guaranteed**. Let's stress-test the algebra to find out
exactly which properties survive and which don't — and *why*.

---

## 3. Build — Core Properties

We build up the algebraic properties of matrices step by step, starting with
the well-behaved ones (addition) and working toward the surprises (multiplication).

### 3.1 Additive Properties

Let $A$, $B$, $C$ be $m \times n$ matrices and $O$ be the $m \times n$ zero matrix.
Matrix addition satisfies all four properties we expect from arithmetic:

| Property | Statement |
|---|---|
| Commutative | $A + B = B + A$ |
| Associative | $(A + B) + C = A + (B + C)$ |
| Additive identity | $A + O = A$ |
| Additive inverse | $A + (-A) = O$ |

Let's verify each with random $3 \times 3$ matrices.

In [ ]:
A = random_matrix(3, 3)
B = random_matrix(3, 3)
C = random_matrix(3, 3)
O = Matrix(3, 3)

print("A ="); print(A)
print("\nB ="); print(B)
print("\nC ="); print(C)

comm = matrices_equal(add(A, B), add(B, A))
print(f"\n1. Commutative  A + B == B + A : {comm}")

assoc = matrices_equal(add(add(A, B), C), add(A, add(B, C)))
print(f"2. Associative  (A+B)+C == A+(B+C) : {assoc}")

ident = matrices_equal(add(A, O), A)
print(f"3. Identity     A + O == A : {ident}")

neg_A = scalar_mul(A, -1)
inv = is_zero_matrix(add(A, neg_A))
print(f"4. Inverse      A + (-A) == O : {inv}")

### 3.2 Multiplicative Properties

Matrix multiplication is more complex. The following properties hold when the
dimensions are compatible (Larson, Theorem 2.4):

| Property | Statement |
|---|---|
| Associative | $A(BC) = (AB)C$ |
| Left distributive | $A(B + C) = AB + AC$ |
| Right distributive | $(A + B)C = AC + BC$ |
| Scalar associativity | $c(AB) = (cA)B = A(cB)$ |

**Note:** Multiplication is **not** commutative in general. $AB \neq BA$ for most matrices.

Let's verify each property with random $3 \times 3$ matrices.

In [ ]:
A = random_matrix(3, 3)
B = random_matrix(3, 3)
C = random_matrix(3, 3)
c = 3.5

assoc = matrices_equal(multiply(A, multiply(B, C)), multiply(multiply(A, B), C))
print(f"1. Associative    A(BC) == (AB)C : {assoc}")

left_dist = matrices_equal(multiply(A, add(B, C)), add(multiply(A, B), multiply(A, C)))
print(f"2. Left distrib   A(B+C) == AB + AC : {left_dist}")

right_dist = matrices_equal(multiply(add(A, B), C), add(multiply(A, C), multiply(B, C)))
print(f"3. Right distrib  (A+B)C == AC + BC : {right_dist}")

lhs = scalar_mul(multiply(A, B), c)
mid = multiply(scalar_mul(A, c), B)
rhs = multiply(A, scalar_mul(B, c))
scal_1 = matrices_equal(lhs, mid)
scal_2 = matrices_equal(lhs, rhs)
print(f"4. Scalar assoc   c(AB) == (cA)B == A(cB) : {scal_1 and scal_2}")

non_comm = not matrices_equal(multiply(A, B), multiply(B, A))
print(f"\nBonus: AB != BA (non-commutative) : {non_comm}")
print("\nAB ="); print(multiply(A, B))
print("\nBA ="); print(multiply(B, A))

### 3.3 The Zero-Matrix Trap

In ordinary arithmetic, if $ab = 0$ then $a = 0$ or $b = 0$. This is the **zero-product property**.

With matrices, this property **fails**. We can find nonzero matrices $A$ and $B$ such that $AB = O$.

**Example.** Let
$$A = \begin{bmatrix} 1 & 2 \\ 2 & 4 \end{bmatrix}, \quad
  B = \begin{bmatrix} -2 & 2 \\ 1 & -1 \end{bmatrix}.$$

Both are nonzero, yet their product is the zero matrix.

In [ ]:
A = Matrix.from_lists([[1, 2], [2, 4]])
B = Matrix.from_lists([[-2, 2], [1, -1]])

AB = multiply(A, B)
print("A ="); print(A)
print("\nB ="); print(B)
print("\nAB ="); print(AB)

print(f"\nA is zero? {is_zero_matrix(A)}")
print(f"B is zero? {is_zero_matrix(B)}")
print(f"AB is zero? {is_zero_matrix(AB)}")
print("\nNeither A nor B is zero, but AB = O.")
print("The zero-product property does NOT hold for matrices!")

### 3.4 Cancellation Failure

In ordinary arithmetic, if $ab = ac$ and $a \neq 0$, we can cancel $a$ to conclude $b = c$.

With matrices, this **cancellation law fails**. We can find $A \neq O$, $B \neq C$
such that $AB = AC$.

**Example.** Let
$$A = \begin{bmatrix} 1 & 2 \\ 2 & 4 \end{bmatrix}, \quad
  B = \begin{bmatrix} 1 & 0 \\ 0 & 1 \end{bmatrix}, \quad
  C = \begin{bmatrix} -1 & -2 \\ 1 & 2 \end{bmatrix}.$$

Then $AB = AC$ even though $B \neq C$. The trick: every column of $B - C$ lies in
the null space of $A$, so $A(B - C) = O$.

In [ ]:
A = Matrix.from_lists([[1, 2], [2, 4]])
B = Matrix.from_lists([[1, 0], [0, 1]])
C = Matrix.from_lists([[-1, -2], [1, 2]])

AB = multiply(A, B)
AC = multiply(A, C)

print("A ="); print(A)
print("\nB ="); print(B)
print("\nC ="); print(C)
print("\nAB ="); print(AB)
print("\nAC ="); print(AC)

print(f"\nAB == AC? {matrices_equal(AB, AC)}")
print(f"B == C?   {matrices_equal(B, C)}")
print("\nAB = AC but B != C. Cancellation fails!")

### 3.5 Why These Break — Preview of Determinants

Both failures share the same root cause: the matrix $A$ is **singular** (non-invertible).

A $2 \times 2$ matrix $\begin{bmatrix} a & b \\ c & d \end{bmatrix}$ is singular when its
**determinant** $ad - bc = 0$. The determinant is the central topic of Chapter 3.

**Key insight (Larson, Sec. 2.3 and Ch. 3):**
- The **zero-product property** holds when $A$ is invertible: if $AB = O$ and $A$ is invertible,
  then $B = A^{-1}O = O$.
- The **cancellation law** holds when $A$ is invertible: if $AB = AC$ and $A$ is invertible,
  then $A^{-1}(AB) = A^{-1}(AC)$, so $B = C$.

When $A$ is singular, it "collapses" information — different inputs can produce the same output.

In [ ]:
A_trap = Matrix.from_lists([[1, 2], [2, 4]])
det_A = det2x2(A_trap)
print(f"A = "); print(A_trap)
print(f"\ndet(A) = (1)(4) - (2)(2) = {det_A}")
print(f"\nA is singular (det = 0): {abs(det_A) < EPSILON}")

print("\n--- Contrast with an invertible matrix ---")
A_good = Matrix.from_lists([[1, 2], [3, 4]])
det_good = det2x2(A_good)
print(f"\nA_good ="); print(A_good)
print(f"\ndet(A_good) = (1)(4) - (2)(3) = {det_good}")
print(f"A_good is invertible (det != 0): {abs(det_good) > EPSILON}")
print("\nWith an invertible matrix, zero-product and cancellation both work.")
print("We will prove this formally in Chapter 3.")

---

## 4. Verify — Systematic Property Checks

Let's stress-test all the properties with random matrices to build confidence
that our implementations are correct.

In [ ]:
print("=" * 60)
print("VERIFICATION: Associativity of Multiplication")
print("=" * 60)

random.seed(100)
for trial in range(5):
    A = random_matrix(3, 3)
    B = random_matrix(3, 3)
    C = random_matrix(3, 3)
    lhs = multiply(A, multiply(B, C))
    rhs = multiply(multiply(A, B), C)
    ok = matrices_equal(lhs, rhs)
    print(f"  Trial {trial + 1}: A(BC) == (AB)C ? {ok}")

print()

In [ ]:
print("=" * 60)
print("VERIFICATION: Distributivity (Left and Right)")
print("=" * 60)

random.seed(200)
for trial in range(5):
    A = random_matrix(3, 3)
    B = random_matrix(3, 3)
    C = random_matrix(3, 3)

    left_lhs = multiply(A, add(B, C))
    left_rhs = add(multiply(A, B), multiply(A, C))
    left_ok = matrices_equal(left_lhs, left_rhs)

    right_lhs = multiply(add(A, B), C)
    right_rhs = add(multiply(A, C), multiply(B, C))
    right_ok = matrices_equal(right_lhs, right_rhs)

    print(f"  Trial {trial + 1}: Left A(B+C)==AB+AC ? {left_ok}  |  Right (A+B)C==AC+BC ? {right_ok}")

print()

In [ ]:
print("=" * 60)
print("VERIFICATION: Zero Product with Non-Zero Matrices")
print("=" * 60)

A = Matrix.from_lists([[1, 2], [2, 4]])
B = Matrix.from_lists([[-2, 2], [1, -1]])
AB = multiply(A, B)

print(f"  A is nonzero: {not is_zero_matrix(A)}")
print(f"  B is nonzero: {not is_zero_matrix(B)}")
print(f"  AB is zero:   {is_zero_matrix(AB)}")
print(f"  Confirmed: nonzero * nonzero = zero.")

print()

In [ ]:
print("=" * 60)
print("VERIFICATION: All Additive + Multiplicative Properties")
print("=" * 60)

random.seed(300)
num_trials = 10
all_pass = True

for trial in range(num_trials):
    A = random_matrix(3, 3)
    B = random_matrix(3, 3)
    C = random_matrix(3, 3)
    O = Matrix(3, 3)
    c = random.uniform(-10, 10)

    checks = {
        "add_comm":       matrices_equal(add(A, B), add(B, A)),
        "add_assoc":      matrices_equal(add(add(A, B), C), add(A, add(B, C))),
        "add_identity":   matrices_equal(add(A, O), A),
        "add_inverse":    is_zero_matrix(add(A, scalar_mul(A, -1))),
        "mul_assoc":      matrices_equal(multiply(A, multiply(B, C)),
                                         multiply(multiply(A, B), C)),
        "left_distrib":   matrices_equal(multiply(A, add(B, C)),
                                         add(multiply(A, B), multiply(A, C))),
        "right_distrib":  matrices_equal(multiply(add(A, B), C),
                                         add(multiply(A, C), multiply(B, C))),
        "scalar_assoc_1": matrices_equal(scalar_mul(multiply(A, B), c),
                                         multiply(scalar_mul(A, c), B)),
        "scalar_assoc_2": matrices_equal(scalar_mul(multiply(A, B), c),
                                         multiply(A, scalar_mul(B, c))),
    }

    failed = [name for name, ok in checks.items() if not ok]
    if failed:
        print(f"  Trial {trial + 1}: FAILED — {failed}")
        all_pass = False
    else:
        print(f"  Trial {trial + 1}: all 9 properties PASSED")

print(f"\nOverall: {'ALL PASSED' if all_pass else 'SOME FAILED'}")

---

## 5. Exercises

Test your understanding with the following problems. Write code in the provided cells.

### Exercise 1 (Easy)

Verify all four **additive properties** (commutative, associative, identity, inverse)
for the specific matrices:

$$A = \begin{bmatrix} 1 & 2 \\ 3 & 4 \end{bmatrix}, \quad
  B = \begin{bmatrix} 5 & 6 \\ 7 & 8 \end{bmatrix}.$$

Print each result and the matrices involved.

In [ ]:
# YOUR CODE HERE


### Exercise 2 (Medium)

A matrix $A$ is called **nilpotent** if $A^k = O$ for some positive integer $k$.
If $A^2 = O$, the matrix is nilpotent of index 2.

Find a $3 \times 3$ **nonzero** matrix $A$ such that $A^2 = O$. Verify by computing
$A^2$ using `multiply(A, A)`.

*Hint:* Think about a matrix where every column is mapped to the null space.
Try a strictly upper-triangular matrix with zeros on and below the main diagonal,
and entries only one step above.

In [ ]:
# YOUR CODE HERE


### Exercise 3 (Challenge)

Write a function `test_cancellation(A, B, C)` that returns `True` if $AB = AC$
and $B \neq C$ (i.e., cancellation fails).

Then, generate 1000 random **singular** $3 \times 3$ matrices and, for each one,
test whether cancellation fails with random $B$ and $C$.

*Strategy for generating singular matrices:* Make row 3 a linear combination of rows 1 and 2.
This guarantees $\det(A) = 0$.

*Strategy for finding $B \neq C$ with $AB = AC$:* Note that $AB = AC$ iff $A(B - C) = O$.
So pick any nonzero $D$ in the null space of $A$, then set $C = B + D$.

In [ ]:
# YOUR CODE HERE


---

## Summary

| Property | Holds for matrices? | Condition |
|---|---|---|
| $A + B = B + A$ | Yes (always) | — |
| $(A + B) + C = A + (B + C)$ | Yes (always) | — |
| $A + O = A$ | Yes (always) | — |
| $A + (-A) = O$ | Yes (always) | — |
| $A(BC) = (AB)C$ | Yes (always) | Dimensions compatible |
| $A(B+C) = AB + AC$ | Yes (always) | Dimensions compatible |
| $(A+B)C = AC + BC$ | Yes (always) | Dimensions compatible |
| $AB = BA$ | **No** (in general) | Only for special matrices |
| $AB = O \Rightarrow A=O$ or $B=O$ | **No** | Requires $A$ invertible |
| $AB = AC \Rightarrow B = C$ | **No** | Requires $A$ invertible |

**Takeaway:** The algebraic properties that *do* hold make matrix algebra powerful.
The ones that *don't* — commutativity, zero-product, cancellation — are
precisely what makes linear algebra richer (and trickier) than scalar arithmetic.
The key concept separating these cases is **invertibility**, which we formalize
in Section 2.3 and connect to **determinants** in Chapter 3.